# Your experiment is randomized. What does its LLM score measure?

A new fee notice may change how people explain a rule. If an LLM rewards the changed writing style, the score contrast combines comprehension with scoring error. This synthetic tutorial uses **existing ppi_py APIs** to estimate the human-reference treatment effect.

Applications include marketing (notice comprehension), enterprise AI (task accuracy), education (conceptual understanding), and LLM evaluation (human-rated quality when independent requests are randomized between models). Same-prompt model comparisons require paired inference rather than this randomized route. Human raters define the outcome; judge scores make the larger sample useful. These are possible applications, not demonstrated deployments.

Install `ppi-python==0.2.3`, `numpy==1.26.4`, and `scipy==1.13.1`. The example generates its data locally and needs no API key.

## 1. Independent labeled and unlabeled experiments

Both cohorts follow the same randomized assignment (probability 0.5). Human scores have mean 50 and SD 10, and treatment changes no human outcome. Judge scores add eight points in arm 1, plus noise with SD 3. The numbers are unbounded illustrative points. Fixed seeds 601 and 602 are not searched for favorable results. Separate IDs make the cohorts disjoint.

In [1]:
"""Human-reference treatment effects for synthetic randomized text outcomes.

Uses upstream APIs, independent labeled/unlabeled IID cohorts, and fixed lam=1.
The conservative contrast interval uses Bonferroni arm intervals. Uses the existing PPI implementation; no model calls or participant data.
"""
import json
import numpy as np
from scipy.stats import norm
from ppi_py import ppi_mean_ci, ppi_mean_pointestimate


def generate_cohort(n, seed, prefix):
    """Simulate an independent IID experiment; the human treatment effect is zero."""
    rng = np.random.default_rng(seed)
    arm = rng.binomial(1, .5, n)
    human = 50 + rng.normal(0, 10, n)
    judge = human + 8*arm + rng.normal(0, 3, n)
    return {'id':np.array([f'{prefix}-{i}' for i in range(n)]),
            'arm':arm, 'human':human, 'judge':judge}




## 2. Estimate each human-reference arm mean with PPI

For each arm, call `ppi_mean_pointestimate` and `ppi_mean_ci` with aligned human labels/predictions from the labeled cohort and predictions **only from the independent unlabeled cohort**. Set `lam=1` in advance for ordinary PPI without power tuning.

The estimate is the unlabeled prediction mean plus the labeled mean residual. Each arm gets a 97.5% asymptotic interval. Subtract bounds as `[lower_1-upper_0, upper_1-lower_0]`: Bonferroni gives a conservative nominal 95% contrast interval when the arm intervals have their stated asymptotic coverage. Coverage is asymptotic rather than an exact finite-sample guarantee.

The human-only baseline uses the same conservative armwise construction. Judge-only has an ordinary two-arm interval for its own score target.

In [2]:
def _scalar(value):
    return np.asarray(value).item()


def run_example(labeled, unlabeled, alpha=.05):
    """Estimate arm means through upstream PPI, then contrast their intervals.

    Both cohorts must be independent IID draws from the same randomized design.
    Human labels and predictions must be aligned within the labeled cohort.
    The unlabeled cohort's human outcomes are never accessed.
    """
    ppi, human_only, judge_only = [], [], []
    for arm in (0,1):
        li = labeled['arm'] == arm
        ui = unlabeled['arm'] == arm
        y, predicted, unpredicted = (labeled['human'][li], labeled['judge'][li], unlabeled['judge'][ui])
        estimate = _scalar(ppi_mean_pointestimate(y,predicted,unpredicted,lam=1))
        lo,hi = ppi_mean_ci(y,predicted,unpredicted,alpha=alpha/2,lam=1)
        ppi.append({'estimate':estimate, 'interval':[_scalar(lo),_scalar(hi)]})
        # Same conservative armwise construction for the human-only comparison.
        half = norm.ppf(1-alpha/4)*np.std(y,ddof=1)/np.sqrt(len(y))
        human_only.append({'estimate':float(y.mean()), 'interval':[float(y.mean()-half),float(y.mean()+half)]})
        judge_only.append(np.concatenate([predicted,unpredicted]))
    def difference(arms):
        return {'estimate':arms[1]['estimate']-arms[0]['estimate'],
                'confidence_interval':[arms[1]['interval'][0]-arms[0]['interval'][1],
                                       arms[1]['interval'][1]-arms[0]['interval'][0]]}
    judge = float(judge_only[1].mean()-judge_only[0].mean())
    half = float(norm.ppf(1-alpha/2)*np.sqrt(sum(np.var(g,ddof=1)/len(g) for g in judge_only)))
    return {'synthetic':True,'population_human_effect':0,'injected_score_bonus':8,
            'labeled_units':len(labeled['arm']),'unlabeled_units':len(unlabeled['arm']),
            'alpha':alpha,'lam':1,'contrast_interval':'Bonferroni combination of armwise asymptotic intervals',
            'ppi':difference(ppi),'human_only':difference(human_only),
            'judge_only':{'estimate':judge,'confidence_interval':[judge-half,judge+half]}}




## 3. Run one fixed draw

Delete the unlabeled human outcomes before analysis. They are simulated for constructing the toy judge, but the estimator cannot access them. A useful judge may reduce the uncertainty relative to human-only inference; that advantage depends on the residual errors.

In [3]:
labeled = generate_cohort(800, 601, "labeled")
unlabeled = generate_cohort(4000, 602, "unlabeled")
assert not set(labeled["id"]) & set(unlabeled["id"])
del unlabeled["human"]  # Never available to the estimator.
result = run_example(labeled, unlabeled)
print("Synthetic example: population human-reference effect = 0")
print("800 labeled units; 4,000 independent unlabeled units")
for key in ("judge_only", "human_only", "ppi"):
    value = result[key]
    low, high = value["confidence_interval"]
    print(f"{key:12s} {value['estimate']:+.2f}  [{low:+.2f}, {high:+.2f}]")


Synthetic example: population human-reference effect = 0
800 labeled units; 4,000 independent unlabeled units
judge_only   +8.08  [+7.50, +8.67]
human_only   -0.73  [-2.85, +1.40]
ppi          +0.46  [-0.77, +1.70]


## 4. Interpret the comparison

![Synthetic experiment: judge-only +8.08; PPI +0.46 with an interval crossing zero.](assets/results.png)

The judge-score interval excludes zero, but its target includes the deliberately injected eight-point scoring bonus. Both human-reference intervals include zero, leaving the direction unresolved; they do not prove equivalence. One draw is not a coverage study or evidence of typical superiority or cost savings. Eight hundred labels are an illustrative input, not a budget recommendation.

The validity argument requires independent IID labeled and unlabeled cohorts from the same population/design, frozen predictions and a specified human rubric, aligned labels/predictions, and sufficient moments/sample sizes for the upstream asymptotic intervals. Conditional grouping by randomized arm preserves the arm distributions. These statements do not cover clustering, nonuniform/adaptive labeling, train-on-evaluation leakage, or reuse of labeled cases in `Yhat_unlabeled`.

Human ratings must measure the outcome the experiment is intended to change. Keep labeled observations out of `Yhat_unlabeled`; this tutorial requires separate IID cohorts.

Methods and attribution: [ppi_py](https://github.com/aangelopoulos/ppi_py), [Prediction-Powered Inference](https://arxiv.org/abs/2301.09633), [DSL](https://naokiegami.com/paper/dsl_ss.pdf).